In [1]:
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D

In [12]:
# =========================
# CONFIG
# =========================
DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"
BASE_MODEL_PATH = r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"
PRUNED_MODEL_PATH = r"D:\Project Jupyter\mobilenetv3_se_attention_pruned_final20_30_40COBADULU"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir   = os.path.join(DATASET_PATH, "val")
test_dir  = os.path.join(DATASET_PATH, "test")

In [14]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FINE_TUNE = 5
PRUNE_STAGES = [0.20, 0.30, 0.40]
CLASSES = ["edible", "poisonous"]

In [16]:
# =========================
# PREPROCESSING
# =========================
def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = test_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = test_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

attention_data = test_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.
Found 2256 images belonging to 2 classes.


In [42]:
# =========================
# LOAD MODEL
# =========================
model = tf.keras.models.load_model(BASE_MODEL_PATH, compile=False)
model.trainable = True

In [43]:
# Backbone = MobileNetV3Large
backbone = model.layers[0]

In [54]:
backbone = tf.keras.applications.MobileNetV3Large(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)


In [64]:
def build_attention_model(backbone):
    se_layers = []

    for layer in backbone.layers:
        if "squeeze_excite" in layer.name.lower():
            se_layers.append(layer)

    if len(se_layers) == 0:
        print("✗ No SE layers found")
        print("DEBUG: contoh 10 nama layer pertama:")
        for l in backbone.layers[:10]:
            print("  ", l.name)
        return None

    print(f"✓ Found {len(se_layers)} SE (squeeze-excite) layers")

    return tf.keras.Model(
        inputs=backbone.input,
        outputs=[l.output for l in se_layers]
    )


In [66]:
for l in backbone.layers:
    if "squeeze" in l.name.lower():
        print(l.name)


expanded_conv_3/squeeze_excite/AvgPool
expanded_conv_3/squeeze_excite/Conv
expanded_conv_3/squeeze_excite/Relu
expanded_conv_3/squeeze_excite/Conv_1
expanded_conv_3/squeeze_excite/Mul
expanded_conv_4/squeeze_excite/AvgPool
expanded_conv_4/squeeze_excite/Conv
expanded_conv_4/squeeze_excite/Relu
expanded_conv_4/squeeze_excite/Conv_1
expanded_conv_4/squeeze_excite/Mul
expanded_conv_5/squeeze_excite/AvgPool
expanded_conv_5/squeeze_excite/Conv
expanded_conv_5/squeeze_excite/Relu
expanded_conv_5/squeeze_excite/Conv_1
expanded_conv_5/squeeze_excite/Mul
expanded_conv_10/squeeze_excite/AvgPool
expanded_conv_10/squeeze_excite/Conv
expanded_conv_10/squeeze_excite/Relu
expanded_conv_10/squeeze_excite/Conv_1
expanded_conv_10/squeeze_excite/Mul
expanded_conv_11/squeeze_excite/AvgPool
expanded_conv_11/squeeze_excite/Conv
expanded_conv_11/squeeze_excite/Relu
expanded_conv_11/squeeze_excite/Conv_1
expanded_conv_11/squeeze_excite/Mul
expanded_conv_12/squeeze_excite/AvgPool
expanded_conv_12/squeeze_excit

In [68]:
# =========================================================
# COMPUTE SE IMPORTANCE
# =========================================================
def compute_se_importance(attention_model, data_gen, steps=20):

    importance = [None] * len(attention_model.outputs)

    for _ in range(steps):
        x, _ = next(data_gen)
        outputs = attention_model(x, training=False)

        for i, out in enumerate(outputs):
            score = tf.reduce_mean(out, axis=0).numpy()
            importance[i] = score if importance[i] is None else importance[i] + score

    return [imp / steps for imp in importance]


In [70]:
def attention_guided_prune(model, backbone, attention_model, data_gen, prune_ratio):

    if attention_model is None:
        print("⚠ Attention model invalid — skipping pruning")
        return model

    se_scores = compute_se_importance(attention_model, data_gen)
    pruned_layers = 0

    for se_layer, score in zip(attention_model.layers[1:], se_scores):

        # Nama block tempat SE berada
        block_name = se_layer.name.split("/")[0]

        # Cari Conv2D dalam block yang sama
        target_convs = [
            l for l in backbone.layers
            if block_name in l.name and isinstance(l, Conv2D)
        ]

        if len(target_convs) == 0:
            continue

        conv = target_convs[-1]  # pointwise conv terakhir
        weights = conv.get_weights()

        if len(weights) == 0:
            continue

        kernel = weights[0]

        if kernel.shape[-1] != len(score):
            continue

        threshold = np.percentile(score, prune_ratio * 100)
        mask = score > threshold

        # 🔥 SOFT PRUNING
        kernel[..., ~mask] = 0

        if len(weights) > 1:
            conv.set_weights([kernel, weights[1]])
        else:
            conv.set_weights([kernel])

        pruned_layers += 1

    print(f"✓ Pruned {pruned_layers} conv layers")
    return model


In [72]:
# =========================================================
# ITERATIVE PRUNING
# =========================================================
for ratio in PRUNE_STAGES:
    print(f"\n===== ITERATIVE PRUNE {int(ratio*100)}% =====")

    attention_model = build_attention_model(backbone)

    model = attention_guided_prune(
        model,
        backbone,
        attention_model,
        attention_data,
        ratio
    )


===== ITERATIVE PRUNE 20% =====
✓ Found 40 SE (squeeze-excite) layers
✓ Pruned 0 conv layers

===== ITERATIVE PRUNE 30% =====
✓ Found 40 SE (squeeze-excite) layers
✓ Pruned 0 conv layers

===== ITERATIVE PRUNE 40% =====
✓ Found 40 SE (squeeze-excite) layers
✓ Pruned 0 conv layers


In [ ]:
# =========================================================
# FINE-TUNING
# =========================================================
model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

start = time.time()

model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_FINE_TUNE,
    verbose=1
)

end = time.time()
print(f"\nTraining time : {(end-start)/60:.2f} minutes")

In [ ]:
model.save(
    PRUNED_MODEL_PATH,
    include_optimizer=False
)


In [ ]:
import os

def get_model_size_mb(model_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)


In [ ]:
base_model_size = get_model_size_mb(BASE_MODEL_PATH)
pruned_model_size = get_model_size_mb(PRUNED_MODEL_PATH)

print(f"Ukuran Baseline Model : {base_model_size:.2f} MB")
print(f"Ukuran Pruned Model   : {pruned_model_size:.2f} MB")

compression_ratio = base_model_size / pruned_model_size
size_reduction = (1 - (pruned_model_size / base_model_size)) * 100

print(f"Compression Ratio : {compression_ratio:.2f}x")
print(f"Size Reduction    : {size_reduction:.2f}%")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
test_data.reset()


In [ ]:

predictions = model.predict(test_data, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes


In [ ]:
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASSES
))


In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


In [ ]:
import time
import numpy as np

def benchmark_inference(model, data_gen, warmup=3, runs=10):
    """
    model      : final pruned model (tanpa pruning wrapper)
    data_gen   : test_data (shuffle=False)
    """

    # Ambil 1 batch
    x_batch, _ = next(data_gen)

    # Warm-up (penting)
    for _ in range(warmup):
        _ = model.predict(x_batch, verbose=0)

    times = []

    for _ in range(runs):
        start = time.time()
        _ = model.predict(x_batch, verbose=0)
        end = time.time()
        times.append(end - start)

    avg_time = np.mean(times)
    per_image_time = avg_time / x_batch.shape[0]

    print("\nInference Benchmark (CPU)")
    print(f"Batch size         : {x_batch.shape[0]}")
    print(f"Avg batch time     : {avg_time:.4f} seconds")
    print(f"Avg per image time : {per_image_time:.6f} seconds")

    return avg_time, per_image_time


In [ ]:
benchmark_inference(model, test_data)
